In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/WIG20.csv")
df = df[df['Data'] < '2025-10']

print(df.head())
print(df.tail())

         Data  Otwarcie  Najwyzszy  Najnizszy  Zamkniecie     Wolumen
0  2022-01-03   2272.18    2302.11    2272.18     2286.50  19199478.0
1  2022-01-04   2293.18    2301.76    2281.04     2300.68  22490984.0
2  2022-01-05   2296.70    2310.97    2283.63     2310.97  23373146.0
3  2022-01-07   2309.61    2314.16    2278.63     2311.93  29717898.0
4  2022-01-10   2311.34    2328.41    2289.59     2291.23  22612381.0
           Data  Otwarcie  Najwyzszy  Najnizszy  Zamkniecie     Wolumen
933  2025-09-24   2802.52    2829.39    2773.28     2828.82  26816327.0
934  2025-09-25   2826.28    2826.28    2788.42     2797.79  33144638.0
935  2025-09-26   2792.34    2832.26    2783.39     2832.26  27472421.0
936  2025-09-29   2845.46    2847.53    2817.46     2832.01  19273923.0
937  2025-09-30   2822.00    2828.33    2803.19     2827.17  26272341.0


In [15]:
df["R_zamkniecie_zamkniecie"] = (df["Zamkniecie"] / df["Zamkniecie"].shift(1)) - 1
df["log_R_zamkniecie_zamkniecie"] = np.log(df["Zamkniecie"] / df["Zamkniecie"].shift(1))

df["R_otwarcie_zamkniecie"] = (df["Zamkniecie"] / df["Otwarcie"]) - 1
df["log_R_otwarcie_zamkniecie"] = np.log(df["Zamkniecie"] / df["Otwarcie"])

df["R_zamkniecie_otwarcie"] = (df["Otwarcie"] / df["Zamkniecie"].shift(1)) - 1
df["log_R_zamkniecie_otwarcie"] = np.log(df["Otwarcie"] / df["Zamkniecie"].shift(1))

df.dropna(inplace=True)
print(df.head())



         Data  Otwarcie  Najwyzszy  Najnizszy  Zamkniecie     Wolumen  \
1  2022-01-04   2293.18    2301.76    2281.04     2300.68  22490984.0   
2  2022-01-05   2296.70    2310.97    2283.63     2310.97  23373146.0   
3  2022-01-07   2309.61    2314.16    2278.63     2311.93  29717898.0   
4  2022-01-10   2311.34    2328.41    2289.59     2291.23  22612381.0   
5  2022-01-11   2300.38    2345.44    2296.03     2344.45  30819446.0   

   R_zamkniecie_zamkniecie  log_R_zamkniecie_zamkniecie  \
1                 0.006202                     0.006182   
2                 0.004473                     0.004463   
3                 0.000415                     0.000415   
4                -0.008954                    -0.008994   
5                 0.023228                     0.022962   

   R_otwarcie_zamkniecie  log_R_otwarcie_zamkniecie  R_zamkniecie_otwarcie  \
1               0.003271                   0.003265               0.002921   
2               0.006213                   0.00619

In [16]:
import scipy.stats as stats
from scipy.stats import jarque_bera, shapiro, normaltest
import matplotlib.pyplot as plt
import seaborn as sns

# Definiujemy kolumny ze stopami zwrotu
return_columns = [
    'R_zamkniecie_zamkniecie', 
    'log_R_zamkniecie_zamkniecie',
    'R_otwarcie_zamkniecie', 
    'log_R_otwarcie_zamkniecie',
    'R_zamkniecie_otwarcie', 
    'log_R_zamkniecie_otwarcie'
]

print("="*80)
print("PODSTAWOWE CHARAKTERYSTYKI STÓP ZWROTU")
print("="*80)

# Obliczanie podstawowych statystyk
statistics_df = pd.DataFrame()

for col in return_columns:
    stats_dict = {
        'Średnia': df[col].mean(),
        'Min': df[col].min(),
        'Max': df[col].max(),
        'Odch. stand.': df[col].std(),
        'Skośność': df[col].skew(),
        'Kurtoza': df[col].kurtosis() + 3
    }
    statistics_df[col] = stats_dict

# Wyświetlanie wyników w czytelnej formie
print(statistics_df.round(6))

print("\n" + "="*80)
print("INTERPRETACJA WYNIKÓW:")
print("="*80)

for col in return_columns:
    print(f"\n{col}:")
    mean_val = df[col].mean()
    std_val = df[col].std()
    skew_val = df[col].skew()
    kurt_val = df[col].kurtosis() + 3
    
    print(f"  • Średnia: {mean_val:.6f} - {'dodatnia' if mean_val > 0 else 'ujemna' if mean_val < 0 else 'zerowa'} średnia stopa zwrotu")
    print(f"  • Zmienność: {std_val:.6f} - {'wysoka' if std_val > 0.02 else 'umiarkowana' if std_val > 0.01 else 'niska'} zmienność")
    print(f"  • Skośność: {skew_val:.4f} - rozkład {'prawostronnie skośny' if skew_val > 0.5 else 'lewostronnie skośny' if skew_val < -0.5 else 'względnie symetryczny'}")
    print(f"  • Kurtoza: {kurt_val:.4f} - {'leptokurtyczny' if kurt_val > 3 else 'platykurtyczny' if kurt_val < 3 else 'mezokurtyczny'} (względem rozkładu normalnego)")

PODSTAWOWE CHARAKTERYSTYKI STÓP ZWROTU
              R_zamkniecie_zamkniecie  log_R_zamkniecie_zamkniecie  \
Średnia                      0.000344                     0.000227   
Min                         -0.108652                    -0.115020   
Max                          0.084365                     0.080995   
Odch. stand.                 0.015333                     0.015358   
Skośność                    -0.169167                    -0.307674   
Kurtoza                      6.813384                     7.329317   

              R_otwarcie_zamkniecie  log_R_otwarcie_zamkniecie  \
Średnia                   -0.000473                  -0.000561   
Min                       -0.058896                  -0.060701   
Max                        0.065091                   0.063060   
Odch. stand.               0.013259                   0.013280   
Skośność                  -0.115206                  -0.183468   
Kurtoza                    4.447296                   4.475900   

       

In [17]:
print("\n" + "="*80)
print("TESTY NORMALNOŚCI ROZKŁADU")
print("="*80)

# Funkcja do interpretacji p-value
def interpret_normality_test(p_value, alpha=0.05):
    if p_value > alpha:
        return f"p={p_value:.6f} > {alpha} → Nie odrzucamy H0 → Rozkład może być normalny"
    else:
        return f"p={p_value:.6f} < {alpha} → Odrzucamy H0 → Rozkład nie jest normalny"

# Przeprowadzanie testów normalności dla każdej stopy zwrotu
for col in return_columns:
    print(f"\n{'-'*60}")
    print(f"TESTY NORMALNOŚCI DLA: {col}")
    print(f"{'-'*60}")
    
    data = df[col].dropna()
    
    # Test 1: Test Shapiro-Wilka
    try:
        shapiro_stat, shapiro_p = shapiro(data)
        print(f"1. Test Shapiro-Wilka:")
        print(f"   Statystyka: {shapiro_stat:.6f}")
        print(f"   {interpret_normality_test(shapiro_p)}")
    except Exception as e:
        print(f"1. Test Shapiro-Wilka: Błąd - {e}")
    
    # Test 2: Test D'Agostino-Pearson
    try:
        dagostino_stat, dagostino_p = normaltest(data)
        print(f"2. Test D'Agostino-Pearson:")
        print(f"   Statystyka: {dagostino_stat:.6f}")
        print(f"   {interpret_normality_test(dagostino_p)}")
    except Exception as e:
        print(f"2. Test D'Agostino-Pearson: Błąd - {e}")
    
    # Test 3: Test Kołmogorowa-Smirnowa
    try:
        # Standaryzacja danych
        standardized_data = (data - data.mean()) / data.std()
        ks_stat, ks_p = stats.kstest(standardized_data, 'norm')
        print(f"3. Test Kołmogorowa-Smirnowa:")
        print(f"   Statystyka: {ks_stat:.6f}")
        print(f"   {interpret_normality_test(ks_p)}")
    except Exception as e:
        print(f"3. Test Kołmogorowa-Smirnowa: Błąd - {e}")


TESTY NORMALNOŚCI ROZKŁADU

------------------------------------------------------------
TESTY NORMALNOŚCI DLA: R_zamkniecie_zamkniecie
------------------------------------------------------------
1. Test Shapiro-Wilka:
   Statystyka: 0.973024
   p=0.000000 < 0.05 → Odrzucamy H0 → Rozkład nie jest normalny
2. Test D'Agostino-Pearson:
   Statystyka: 92.470614
   p=0.000000 < 0.05 → Odrzucamy H0 → Rozkład nie jest normalny
3. Test Kołmogorowa-Smirnowa:
   Statystyka: 0.037204
   p=0.145724 > 0.05 → Nie odrzucamy H0 → Rozkład może być normalny

------------------------------------------------------------
TESTY NORMALNOŚCI DLA: log_R_zamkniecie_zamkniecie
------------------------------------------------------------
1. Test Shapiro-Wilka:
   Statystyka: 0.970517
   p=0.000000 < 0.05 → Odrzucamy H0 → Rozkład nie jest normalny
2. Test D'Agostino-Pearson:
   Statystyka: 112.335114
   p=0.000000 < 0.05 → Odrzucamy H0 → Rozkład nie jest normalny
3. Test Kołmogorowa-Smirnowa:
   Statystyka: 0.03